# Sprint 4 — CNN
## Classificação de Ambientes por Imagem com Rede Neural Convolucional

**Integrantes:**
- Kaio Vinicius Meireles Alves — RM553282
- Lucas Alves de Souza — RM553956
- Lucas de Freitas Pagung — RM553242
- Guilherme Fernandes de Freitas — RM554323
- João Pedro Chizzolini de Freitas — RM553172

## 1. Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import numpy as np
import os

print("TF version:", tf.__version__)
print("GPU disponível:", tf.config.list_physical_devices('GPU'))

## 2. Download do Dataset

In [ ]:
import os

os.environ['KAGGLE_USERNAME'] = 'lucassouza777'
os.environ['KAGGLE_KEY'] = 'KGAT_22df4b83aa79e03e12466375b8706b3c'

!pip install kaggle -q
!kaggle datasets download -d robinreni/house-rooms-image-dataset -p /content/dataset --unzip

## 3. Verificação do Dataset

In [ ]:
base = "/content/dataset/House_Room_Dataset"

for cls in os.listdir(base):
    cls_path = os.path.join(base, cls)
    if os.path.isdir(cls_path):
        count = len(os.listdir(cls_path))
        print(f"{cls}: {count} imagens")

## 4. Pré-processamento e Data Augmentation

In [ ]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

train_gen = datagen.flow_from_directory(
    base,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    seed=42
)

val_gen = datagen.flow_from_directory(
    base,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    seed=42
)

print("Classes:", train_gen.class_indices)
print("Treino:", train_gen.samples, "imagens")
print("Validação:", val_gen.samples, "imagens")

## 5. Modelo — MobileNetV2 com Transfer Learning

In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(5, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 6. Treinamento

In [ ]:
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=callbacks
)

## 7. Avaliação

In [ ]:
loss, acc = model.evaluate(val_gen)
print(f"Loss: {loss:.4f}")
print(f"Acurácia: {acc:.4f}")

## 8. Matriz de Confusão

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

val_gen.reset()
y_pred = model.predict(val_gen)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = val_gen.classes
classes = list(val_gen.class_indices.keys())

cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes, cmap='Blues')
plt.title('Matriz de Confusão')
plt.ylabel('Real')
plt.xlabel('Predito')
plt.tight_layout()
plt.savefig('matriz_confusao.png', dpi=150)
plt.show()

print(classification_report(y_true, y_pred_classes, target_names=classes))

## 9. Curvas de Treino

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Treino')
plt.plot(history.history['val_accuracy'], label='Validação')
plt.title('Acurácia')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Treino')
plt.plot(history.history['val_loss'], label='Validação')
plt.title('Loss')
plt.legend()

plt.tight_layout()
plt.savefig('curvas_treino.png', dpi=150)
plt.show()